In [167]:
from ET_pylib.setup import *
setup_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Kernekraftstød: numerisk analyse

Tabeller for basisforløbet mod kernekraftforløbet i GrønREFORM, hvor ny kernekraft i DK1 og DK2 erstatter havvind. Velfærdsåret er 2040.

Notebooken har to dele: **elmarkedet** (afsnit 3–12) og **fjernvarmemarkedet** (afsnit 13–18). Stødet vælges i afsnit 1: kernekraft som kondensværk eller som udtagsværk, der også leverer fjernvarme.

*`1. opsætning`*

*vælg geografi, år og kørsel*

In [168]:
# 1. opsætning
GEO   = "disaggEA"
print("Kernekraft-id'er:", ", ".join(f"{i} (fra {y})" for i, y in use_geography(GEO).items()))

yr    = 2040
ids   = NUCLEAR_IDS_BY_GEO[GEO]
RUN   = "disaggEA_2040_kf25om"           # 2040 med brintfiks
# SHOCK = "shock_nuc_2040.pkl"                # kondens (CD)
SHOCK = "shock_nuc_chp_2040.pkl"          # kraftvarme (EX), fire varmeområder
# RUN   = "disaggEA_2025_2050"           # DREAMs 2025-2050 uden brintfiks

Kernekraft-id'er: idNUC_DK1 (fra 2035), idNUC_DK2 (fra 2035), idNUC_DK1_C (fra 2035), idNUC_DK1_D (fra 2035), idNUC_DK2_C (fra 2035), idNUC_DK2_D (fra 2035)


*`2. data`*

*indlæs basis- og stødforløb og vis kapacitetsbyttet*

In [169]:
# 2. data
baseline_output, shock_output = load_run(RUN, SHOCK)
nuc, cut = nuc_wind_swap(baseline_output, shock_output, yr=yr, nuc_ids=ids)

Kapacitetsbytte — 2040 (GW)
                                  basis  stød      Δ
────────────────────────────────────────────────────
Kernekraft DK1_C                   0,00  0,35  +0,35
Kernekraft DK1_D                   0,00  0,30  +0,30
Kernekraft DK2_C                   0,00  0,25  +0,25
Kernekraft DK2_D                   0,00  0,10  +0,10
Havvind DK1                        4,50  3,46  -1,04
Havvind DK2                        1,79  1,17  -0,62
────────────────────────────────────────────────────
Kernekraft i alt                               +1,00
Havvind i alt                                  -1,67
Kernekraft, el i modtryk (GWe)                 +0,85
Kernekraft, varme (GW-th)                      +0,48
────────────────────────────────────────────────────
Havvind fjernet pr. GW kernekraft: 1,67



*byg timebalancerne pr. zone og for Danmark samlet*

In [170]:
ctx_el     = build_el_balance_data(baseline_output, shock_output, years=yr)
ctx_el_agg = build_el_balance_data_country(baseline_output, shock_output, years=yr)
shares     = build_supply_shares(ctx_el, aggregates=c_aggs_countries)

## Elmarkedet

*`3. priser og knaphed`*

*efterspørgselsvægtede prismomenter og andelen af nulpris- og mangeltimer*

In [171]:
# 3. priser og knaphed
print_price_summary(ctx_el, eas=("DK1", "DK2"), countries=("DK",), years=yr)

Priser og knaphed — 2040
                                Pris (kr./MWh)             Fordeling        Timer (pct.)
                          gns.  min.    maks.  std.afv.  skævhed  kurtose  nulpris  mangel
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Vest), basis    383,9  40,4  1.179,9     277,2     1,19     0,48      0,0    13,7
Danmark (Vest), stød     381,6  39,5  1.164,8     272,0     1,18     0,47      0,0    13,7
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Øst), basis     381,0  35,4  1.184,5     271,3     1,21     0,54      0,0    15,9
Danmark (Øst), stød      378,8  33,5  1.155,3     265,4     1,18     0,51      0,0    14,3
──────────────────────────────────────────────────────────────────────────────────────────
Danmark, basis           382,9  38,7  1.169,1     274,9     1,20     0,50      0,0    13,7
Danmark, stød            380,6  37,4  1.145,8     269,5     1,18   

*`4. handel`*

*eksport, import og nettoeksport pr. handelspartner for DK, NO og SE*

In [172]:
# 4. handel
print_trade_summary(ctx_el, countries=("DK"), years=yr)

Handel — Danmark, 2040
                           Eksport (TWh)  Import (TWh)      Netto (TWh)         Netto (mio. kr.)
                            basis    stød  basis   stød  basis   stød      Δ   basis    stød       Δ
────────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,50   16,68   4,49   3,70  12,02  12,98  +0,96   3.362   4.202    +840
Storbritannien og Irland     3,75    3,87   4,00   4,06  -0,25  -0,19  +0,06    -397    -277    +120
Nederlandene                 1,07    1,14   1,50   1,52  -0,43  -0,38  +0,05     -82       8     +90
Norge                        1,53    1,48   8,79   8,85  -7,26  -7,36  -0,10  -1.499  -1.523     -24
Sverige                      3,02    3,01   7,64   7,68  -4,61  -4,67  -0,06  -2.363  -2.339     +24
────────────────────────────────────────────────────────────────────────────────────────────────────
I alt                       25,87   26,18  26,42  25,81  -0,53   0,38  +

*bytteforhold pr. handelspartner: eksportpris over importpris*

In [173]:
print_terms_of_trade_partner(ctx_el, country="DK", years=yr)

Bytteforhold pr. handelspartner — Danmark, 2040
                           Eksport (TWh)  Import (TWh)  Eksportpris   Importpris          Bytteforhold
                            basis    stød  basis   stød  basis  stød  basis  stød  basis   stød       Δ  Δ pct.
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,50   16,68   4,49   3,70    325   339    446   394  0,729  0,862  +0,132   +18,1
Storbritannien og Irland     3,75    3,87   4,00   4,06    285   297    367   351  0,778  0,845  +0,067    +8,7
Nederlandene                 1,07    1,14   1,50   1,52    413   448    350   331  1,181  1,354  +0,172   +14,6
Norge                        1,53    1,48   8,79   8,85    124   123    192   193  0,646  0,638  -0,008    -1,2
Sverige                      3,02    3,01   7,64   7,68    119   117    356   351  0,334  0,335  +0,001    +0,2
─────────────────────────────────────────────────────────────────

*`5. fangstpris og værdifaktor`*

*produktionsvægtet fangstpris pr. anlægstype i Danmark*

In [174]:
# 5. fangstpris og værdifaktor
print_capture(capture_summary(ctx_el, country="DK", years=yr), year=yr)

Fangstpris og værdifaktor, el — Danmark, 2040
                                         Produktion (TWh)     Fangstpris (kr./MWh)   Værdifaktor
                                        basis   stød      Δ    basis     stød      Δ            Δ
─────────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,02  33,75  -7,27    298,8    299,4   +0,6        +0,01
Solceller                               19,01  19,01  -0,00    189,1    186,3   -2,7        -0,00
Landvind                                13,96  13,96  +0,01    275,9    277,9   +2,0        +0,01
Modtryksværk (kraftvarme)                1,61   1,59  -0,02    557,7    556,9   -0,8        +0,01
Ellagring                                1,51   1,52  +0,00    655,7    644,2  -11,5        -0,02
Modtryksværk med bypass (kraftvarme)     0,44   0,44  -0,00    818,5    785,6  -32,9        -0,07
Udtagsværk (kraftvarme)                  0,40   0,37  -0,03    884,4    8

*`6. kapacitetsfaktorer`*

*realiseret produktion over installeret kapacitet gange 8760 timer*

In [175]:
# 6. kapacitetsfaktorer
print_capacity_factor(capacity_factor_summary(ctx_el, country="DK", years=yr), year=yr)

Kapacitetsfaktor, el — Danmark, 2040
                                         Produktion (TWh)   Kapacitetsfaktor (pct., ECap)
                                        basis   stød      Δ      basis      stød         Δ
──────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,02  33,75  -7,27       50,4      50,5      +0,1
Solceller                               19,01  19,01  -0,00       12,1      12,1      -0,0
Landvind                                13,96  13,96  +0,01       27,7      27,7      +0,0
Modtryksværk (kraftvarme)                1,61   1,59  -0,02       26,9      26,6      -0,3
Ellagring                                1,51   1,52  +0,00        7,6       7,6      +0,0
Modtryksværk med bypass (kraftvarme)     0,44   0,44  -0,00       16,1      16,1      -0,0
Udtagsværk (kraftvarme)                  0,40   0,37  -0,03       13,4      12,6      -0,8
Eksogen elproduktion                     0,06   0,06  

*`7. forbrugeroverskud`*

*ΔCS pr. zone som trapez på DC-IIA-efterspørgslen, $\Delta CS = -\sum_h \left[D_{0h} + \tfrac{1}{2}(D_{1h} - D_{0h})\right]\Delta p_h$*

In [176]:
# 7. forbrugeroverskud
print_cs_table(baseline_output, shock_output, year=yr)

Forbrugeroverskud, el — 2040 (mio. kr.)
                                 heraf
                         ΔCS    pris  mængde
────────────────────────────────────────────
Danmark (Vest)         +94,1   +94,1    +0,1
Danmark (Øst)          +47,1   +47,0    +0,1
Danmark (Bornholm)      +0,0    +0,0    +0,0
Danmark i alt         +141,2  +141,1    +0,1
────────────────────────────────────────────
Norge (syd)            -35,9   -35,9    +0,0
Norge (nord)            -4,4    -4,4    +0,0
Norge (midt)            -7,5    -7,5    +0,0
Norge i alt            -47,8   -47,8    +0,0
────────────────────────────────────────────
Sverige (nord)          -2,3    -2,3    +0,0
Sverige (midtnord)      -4,1    -4,1    +0,0
Sverige (midtsyd)      -17,5   -17,5    +0,0
Sverige (syd)          +15,7   +15,7    +0,0
Sverige i alt           -8,2    -8,2    +0,0
────────────────────────────────────────────
I alt                  +85,2   +85,1    +0,1
────────────────────────────────────────────



*`8. producentoverskud`*

*UP gange kapacitet pr. anlægstype, opdelt i pris-, mængde- og resteffekt*

In [177]:
# 8. producentoverskud
areas = c_aggs["DK"]
maps = dict(id2Pt=baseline_output["id2Pt"].vals, id2EA=shock_output["id2EA"].vals)
dec_maps = dict(**maps,
                id2HA=shock_output["id2HA"].vals,
                id2cIdxE=shock_output["id2cIdxE"].vals,
                id2cIdxH=shock_output["id2cIdxH"].vals)

bt = producer_profit(baseline_output, shock_output, areas=areas, year=yr,
                     split_nuclear=True, **maps)
dec = profit_decomposition(baseline_output, shock_output, areas=areas, year=yr,
                           split_nuclear=True, **dec_maps)
print_producer_profit(bt, year=yr, areas=areas, decomp=dec)

Producentoverskud — Danmark, 2040 (mio. kr.)
                                            Producentoverskud                Δ, heraf
                                           basis      stød         Δ    pris  mængde      rest
──────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                  6.962,9   5.758,1  -1.204,8   +74,1    -7,3  -1.271,6
Solceller                                3.136,2   3.065,2     -71,0   -72,1    +0,0      +1,0
Varmekedel                                -542,3    -611,1     -68,8   -79,6    +6,0      +4,7
Eksogen fjernvarmeproduktion               471,3     439,1     -32,1   -33,3    +0,2      +1,0
Ellagring                                  510,7     491,8     -18,9   -20,0    +0,9      +0,2
Udtagsværk (kraftvarme)                    126,3     113,6     -12,7   -13,0    +0,3      -0,0
Modtryksværk med bypass (kraftvarme)      -182,4    -194,1     -11,7   -32,0    +1,4     +18,9
Solvarme      

*`9. produktion`*

*elproduktion pr. anlægstype i Danmark*

In [178]:
# 9. produktion
by_qty = producer_generation(baseline_output, shock_output, areas=areas, year=yr,
                             split_nuclear=True, **maps)
print_producer_qty(by_qty, year=yr, areas=areas)

Elproduktion — Danmark, 2040 (TWh)
                                        basis   stød      Δ
───────────────────────────────────────────────────────────
Havvind                                 41,02  33,75  -7,27
Udtagsværk (kraftvarme)                  0,40   0,37  -0,03
Modtryksværk (kraftvarme)                1,61   1,59  -0,02
Solceller                               26,03  26,03  -0,00
Eksogen elproduktion                     0,06   0,06  -0,00
Kondensværk                              0,00   0,00  -0,00
Modtryksværk med bypass (kraftvarme)     0,44   0,44  -0,00
Vandkraft (flodkraft)                    0,02   0,02  +0,00
Landvind                                13,96  13,96  +0,01
Ellagring                                4,20   4,21  +0,01
Kernekraft                               0,00   7,30  +7,30
───────────────────────────────────────────────────────────
I alt                                   87,73  87,74  +0,01



*`10. flaskehalsindtægt`*

*flaskehalsindtægt til de danske TSO'er; hver linjes rente deles ligeligt mellem de to ender*

In [179]:
# 10. flaskehalsindtægt
print_ci(baseline_output, shock_output, year=yr, country="DK")

Flaskehalsindtægt til TSO — Danmark, 2040 (mio. kr.)
                        basis     stød      Δ
─────────────────────────────────────────────
Danmark (Vest)        2.060,1  2.017,3  -42,8
Danmark (Øst)           402,1    388,8  -13,3
Danmark (Bornholm)      185,6    176,5   -9,1
─────────────────────────────────────────────
I alt                 2.647,8  2.582,6  -65,2



*`11. velfærd`*

*værdien af kernekraftens fleksibilitet: værkerne tvinges op på minimumslast, og den ekstra produktion værdisættes til modellens priser*
$$
    \Delta W = \sum_{h\in\mathcal H} (p_h-MC-s) \cdot \Delta q_h
$$

In [180]:
# 11. velfærd
print_flex(shock_output, year=yr, u_mins=(0.3))      # must-run og 30 pct. minimumslast

Fleksibilitetsværdi, kernekraft — 2040, minimumslast 30 pct.
                          TWh            kr./MWh     mio. kr.
                    produktion  ekstra  pris, ekstra        ΔW
──────────────────────────────────────────────────────────────
Kernekraft DK1_C          2,58    0,02          56,3      -1,5
Kernekraft DK1_D          2,18    0,01          59,8      -0,6
Kernekraft DK2_C          1,83    0,01          53,8      -1,1
Kernekraft DK2_D          0,71    0,00          49,8      -0,3
──────────────────────────────────────────────────────────────
I alt                     7,30    0,04          55,7      -3,4
──────────────────────────────────────────────────────────────



*samlet velfærdseffekt for Danmark ved tre rentestrukturer*

In [181]:
blocks = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E",),
)
print_welfare_blocks(blocks, regions="DK")

Velfærdseffekt — Danmark, 2040 (mio. kr./år)
                                       FM     4 pct.     6 pct.
───────────────────────────────────────────────────────────────
Δ Forbrugeroverskud               +141,19    +141,19    +141,19
Δ Producentoverskud               +537,87    +537,87    +537,87
Δ Flaskehalsindtægter              -65,20     -65,20     -65,20
Annuitet, netto                   -814,79  -1.056,15  -1.891,60
Systemomkostning, VE            +1.053,28  +1.053,28  +1.053,28
Systemomkostning, kernekraft      -220,37    -220,37    -220,37
───────────────────────────────────────────────────────────────
Netto velfærdseffekt              +631,97    +390,61    -444,84
───────────────────────────────────────────────────────────────
Annuitet, kernekraft (ny)       +2.504,38  +2.853,22  +4.149,17
Annuitet, havvind (nedlagt)     -1.689,59  -1.797,07  -2.257,57
───────────────────────────────────────────────────────────────
Annuitet: + = omkostning, - = sparet.
Systemomkostning, VE:

*`12. LCOE`*

*LCOE af GR's egne drifts- og produktionstal plus annuitet af investeringen*

In [182]:
# 12. LCOE
print_lcoe(shock_output, year=yr, r=0.04)

LCOE — Danmark, 2040, rente: 4,0 pct. (øre/kWh)
               kap.faktor (pct.)  annuitet   FOM    MC  LCOE
────────────────────────────────────────────────────────────
Kernekraft                  83,4      55,2  13,7  12,6  81,4
Havvind                     50,5      24,4   9,0   4,0  37,3
Landvind                    27,7      23,1   7,8   2,1  33,0
Solceller                   12,2      20,7   7,9   0,0  28,6
────────────────────────────────────────────────────────────
VE (vægtet)                 22,4      22,8   8,4   2,2  33,4
────────────────────────────────────────────────────────────



---

## Fjernvarmemarkedet

Fra her gælder figurer og tabeller fjernvarmen i de fire danske varmeområder (central og decentral i Vest- og Østdanmark). Handel og flaskehalsindtægt findes ikke på fjernvarmemarkedet.

*`13. data`*

*byg fjernvarmebalancerne pr. varmeområde*

In [183]:
# 13. data
ctx_dh = build_el_balance_data(baseline_output, shock_output, years=yr, market="H")

*`14. priser og knaphed`*

*fjernvarmepriser og knaphed pr. varmeområde og for Danmark*

In [184]:
# 14. priser og knaphed
print_price_summary(ctx_dh, eas=HA_AGGS["DK"], countries=("DK",), years=yr)

Fjernvarmepriser og knaphed — 2040
                                       Pris (kr./MWh)            Fordeling        Timer (pct.)
                                  gns.  min.  maks.  std.afv.  skævhed  kurtose  nulpris  mangel
────────────────────────────────────────────────────────────────────────────────────────────────
Vestdanmark, central, basis      216,5  62,6  372,4      79,3    -0,04    -0,79      0,0     0,0
Vestdanmark, central, stød       197,5  61,5  358,4      76,2     0,36    -0,56      0,0     0,0
────────────────────────────────────────────────────────────────────────────────────────────────
Vestdanmark, decentral, basis    329,7  69,6  557,5     116,5    -0,86    -0,00      0,0     0,0
Vestdanmark, decentral, stød     310,1  66,5  489,6     110,7    -0,92    -0,32      0,0     0,0
────────────────────────────────────────────────────────────────────────────────────────────────
Østdanmark, central, basis       318,3  61,6  811,0     132,6     1,29     4,86      0,0     1

*`15. fangstpris og kapacitetsfaktor`*

*fangstpris og kapacitetsfaktor for fjernvarme pr. anlægstype i Danmark*

In [185]:
# 15. fangstpris og kapacitetsfaktor
print_capture(capture_summary(ctx_dh, country="DK", years=yr), year=yr, market="H")
print_capacity_factor(capacity_factor_summary(ctx_dh, country="DK", years=yr), year=yr,
                      market="H")

Fangstpris og værdifaktor, fjernvarme — Danmark, 2040
                                         Produktion (TWh)   Fangstpris (kr./MWh)  Værdifaktor
                                        basis   stød      Δ  basis   stød       Δ            Δ
──────────────────────────────────────────────────────────────────────────────────────────────
Elkedler og varmepumper                 28,49  26,41  -2,08  267,5  253,3   -14,2        +0,01
Modtryksværk (kraftvarme)                7,86   7,74  -0,12  334,7  317,5   -17,2        +0,01
Varmelagring                             5,44   5,45  +0,01  241,0  231,2    -9,8        +0,02
Varmekedel                               4,50   3,90  -0,60  349,3  326,5   -22,8        -0,01
Eksogen fjernvarmeproduktion             2,06   2,05  -0,01  244,5  227,9   -16,6        -0,01
Modtryksværk med bypass (kraftvarme)     1,90   1,81  -0,09  358,4  345,4   -12,9        +0,03
Solvarme                                 0,87   0,87  -0,00  179,3  166,9   -12,4        -0,

*`16. forbrugeroverskud`*

*forbrugeroverskud på fjernvarme pr. varmeområde*

In [186]:
# 16. forbrugeroverskud
print_cs_table(baseline_output, shock_output, year=yr, market="H")

Forbrugeroverskud, fjernvarme — 2040 (mio. kr.)
                                     heraf
                             ΔCS    pris  mængde
────────────────────────────────────────────────
Vestdanmark, central      +265,5  +264,3    +1,3
Vestdanmark, decentral    +233,3  +232,5    +0,8
Østdanmark, central       +236,4  +235,7    +0,7
Østdanmark, decentral      +72,7   +72,5    +0,1
Danmark i alt             +807,9  +805,0    +2,9
────────────────────────────────────────────────
I alt                     +807,9  +805,0    +2,9
────────────────────────────────────────────────



*`17. produktion`*

*fjernvarmeproduktion pr. anlægstype i Danmark*

In [187]:
# 17. produktion
areas_dh = HA_AGGS["DK"]
by_qty_dh = producer_generation(baseline_output, shock_output, areas=areas_dh, year=yr,
                                split_nuclear=True, market="H", **maps)
print_producer_qty(by_qty_dh, year=yr, areas=areas_dh, market="H")

Fjernvarmeproduktion — Danmark, 2040 (TWh)
                                        basis   stød      Δ
───────────────────────────────────────────────────────────
Elkedler og varmepumper                 28,49  26,41  -2,08
Varmekedel                               4,50   3,90  -0,60
Modtryksværk (kraftvarme)                7,86   7,74  -0,12
Modtryksværk med bypass (kraftvarme)     1,90   1,81  -0,09
Udtagsværk (kraftvarme)                  0,67   0,62  -0,04
Eksogen fjernvarmeproduktion             2,06   2,05  -0,01
Power-to-X med varme                     0,44   0,44  -0,00
Solvarme                                 0,87   0,87  -0,00
Varmelagring                             5,44   5,45  +0,01
Kernekraft                               0,00   3,24  +3,24
───────────────────────────────────────────────────────────
I alt                                   52,22  52,53  +0,31



*`18. velfærd`*

*velfærd med både el- og fjernvarmemarkedet (ΔCS inkl. fjernvarme)*

In [197]:
# 18. velfærd
blocks_eh = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E", "H"),
)
print_welfare_blocks(blocks_eh, regions="DK")

Velfærdseffekt — Danmark, 2040 (mio. kr./år)
                                       FM     4 pct.     6 pct.
───────────────────────────────────────────────────────────────
Δ Forbrugeroverskud               +949,07    +949,07    +949,07
Δ Producentoverskud               +537,87    +537,87    +537,87
Δ Flaskehalsindtægter              -65,20     -65,20     -65,20
Annuitet, netto                 -1.190,45  -1.484,14  -2.513,98
Systemomkostning, VE            +1.053,28  +1.053,28  +1.053,28
Systemomkostning, kernekraft      -220,37    -220,37    -220,37
───────────────────────────────────────────────────────────────
Netto velfærdseffekt            +1.064,19    +770,50    -259,34
───────────────────────────────────────────────────────────────
Annuitet, kernekraft (ny)       +2.880,04  +3.281,21  +4.771,54
Annuitet, havvind (nedlagt)     -1.689,59  -1.797,07  -2.257,57
───────────────────────────────────────────────────────────────
Annuitet: + = omkostning, - = sparet.
Systemomkostning, VE: